# Pattern 04: Cross-encoder reranking (BM25 base)

Follows SPEC.md §8's mandatory 8-section template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for the generation step and
`MockReranker` for reranking (via `get_reranker()`), matching every other pattern's mock/real
split -- no real model download is required just to view or re-run this notebook as committed.

Unlike patterns 03/05/06, **section 7 below is 100% real**, not a PENDING placeholder. This
pattern is built on BM25 (deterministic, no API key) plus a local cross-encoder
(`BAAI/bge-reranker-v2-m3`, also no API key) -- both retrieval components run for free, with no
external dependency on `OPENAI_API_KEY`. The real findings were gathered via a one-off script
(`real_rerank_check.py`, not committed to this repo) run directly against the real pilot corpus
and `evals/qa_set.jsonl`, with `sentence-transformers` installed (see `pyproject.toml`). Only the
generated answer *text* and the judge scores in sections 4-6 below are mocked.


## Reproducibility header (SPEC.md §11)

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: 612a622ff86805e202bef0505ca1f2b5a4fefd82


## Setup (loaded once, used by every section below)

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import MockLLM, get_llm

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm


## 1. What this pattern does

Cross-encoder reranking is a two-stage retrieval pipeline: a cheap first-stage retriever (here,
BM25, `recipes/bm25.py`'s `BM25Index`) pulls a wider candidate pool (top-20, `CANDIDATES_K` in
`recipes/rerank.py`), then a cross-encoder model (`BAAI/bge-reranker-v2-m3` via
`sentence-transformers`) scores each `(query, candidate)` pair *jointly* -- unlike a bi-encoder
(dense embeddings), which scores query and document independently and compares vectors afterward,
a cross-encoder sees both texts together and can model fine-grained interactions between them. The
top-k after reranking is what gets passed to generation. This costs latency (one forward pass per
candidate, all local) but no API money.


## 2. When to use it

- You already have a working first-stage retriever (BM25 or dense) and want a quality boost with
  no added API cost -- the reranker is a local model, so this is pure compute, not $/query
- Your first-stage retriever's top-1 to top-3 quality matters a lot (reranking only reorders what
  the first stage already surfaced -- see section 7's "ceiling" limitation below)
- You can afford the extra local inference latency (loading + running a cross-encoder over ~20
  candidates per query)


## 3. When NOT to use it

- The first-stage retriever's candidate pool doesn't contain the relevant chunk at all -- reranking
  can only reorder what's already in the top-`CANDIDATES_K`, it cannot retrieve something BM25
  missed entirely (see section 7's q13 finding, where the relevant chunks *were* recoverable
  because they were present, if buried, in BM25's top-20)
- Query latency budget can't absorb loading and running a local cross-encoder model
- The first-stage retriever is already reliably ranking the right chunk at rank 1 -- reranking adds
  cost for no benefit, and (see section 7's q16 finding) can occasionally *demote* an
  already-correct top-1 result


## 4. Implementation

In [3]:
from recipes.rerank import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("What does PCEval stand for?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)


retrieved: ['arxiv:2601.00129#2', 'arxiv:2601.00904#1', 'arxiv:2601.00086#1']
answer: This is a mock response.


## 5. Run on our eval set

In [4]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="04_rerank",
    judges_enabled=True,
)


=== 04_rerank (n=18) ===
  hit@3: 0.222  [95% CI 0.056, 0.444]
  hit@10: 0.611  [95% CI 0.389, 0.833]
  mrr: 0.162  [95% CI 0.094, 0.235]
  faithfulness: 1.000  [95% CI 1.000, 1.000]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 1.000  [95% CI 1.000, 1.000]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 1.8
  p95_latency_ms: 4.4
  usd_per_query: $0.01071
  eval_usd: $0.1928


## 6. Example query walkthrough

One example per eval-set category, showing the reranked chunks and the (mocked) final answer.
Under `MockReranker`, reordering is deterministic-but-arbitrary (hash-based), so which chunks come
back is not meaningful here -- section 7 below uses the real cross-encoder instead.

In [5]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print()


--- keyword ---
Q: What does PCEval stand for?
Retrieved: ['arxiv:2601.00129#2', 'arxiv:2601.00904#1', 'arxiv:2601.00086#1']
A: This is a mock response.

--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
Retrieved: ['arxiv:2601.00904#1', 'arxiv:2601.00090#0', 'arxiv:2601.00086#1']
A: This is a mock response.

--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
Retrieved: ['arxiv:2601.00121#0', 'arxiv:2601.00097#2', 'arxiv:2601.00086#1']
A: This is a mock response.

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
Retrieved: ['arxiv:2601.00121#1', 'arxiv:2601.00138#0', 'arxiv:2601.11580#0']
A: This is a mock response.



## 7. Where this pattern FAILS (and succeeds)

Real findings, computed via a one-off script running real BM25 retrieval (top-20 candidates) and a
real `BAAI/bge-reranker-v2-m3` cross-encoder against the pilot corpus + `evals/qa_set.jsonl`. Zero
API cost -- both components are local models, no `OPENAI_API_KEY` needed. Full per-question output
is in `tasks/todo.md`'s P3 section.

**Success 1 -- q13 (multi-hop), BM25's worst failure gets substantially better:** `02_bm25.ipynb`
section 7 documents this as BM25's hardest failure -- hit@3 = 0.0, with one of the two needed
chunks (`arxiv:2601.00129#0`, `arxiv:2601.00130#0`) buried at BM25 ranks 13 and 10 respectively.
Reranking promotes them to **ranks 3 and 4**: both chunks *were* present in BM25's top-20 candidate
pool, just ranked too low for a top-3 or top-5 cutoff to catch, and the cross-encoder's joint
query-document scoring recognizes their relevance far better than BM25's pure term-overlap. This is
reranking's core value proposition working as expected -- but note it only works *because* both
chunks survived into the top-20 candidate pool in the first place (see the "when NOT to use"
limitation above).

**Success 2 -- q06 (keyword) and q09 (paraphrase), reranking fixes a "right paper, wrong section"
error:** `02_bm25.ipynb` section 7 separately documents BM25 preferring a paper's methods section
over its abstract when the methods section has denser keyword overlap. The cross-encoder corrects
this: for q06 (*"What architecture powers Mathesis's mathematical reasoning?"*), the correct chunk
`arxiv:2601.00125#0` moves from BM25 rank 3 to **reranked rank 1**; for q09 (the tool-use
adaptation question), `arxiv:2601.00086#0` moves from BM25 rank 2 to **reranked rank 1**. The
cross-encoder's joint scoring is better at recognizing "this passage directly answers the
question" than BM25's density-only signal.

**Failure -- q16 (filter), reranking demotes an already-correct top-1 result:** For *"Among the
cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep
learning?"*, BM25 alone already ranks the correct chunk `arxiv:2601.00907#0` at **rank 1**.
Reranking pushes it down to **rank 2**, swapped out by a chunk the cross-encoder judged more
relevant to the surface wording of the query. This is a genuine downside, not a hypothetical one:
reranking is not monotonically an improvement over the base retriever -- it can occasionally
overrule a first-stage ranker that was already right, and (per the "when NOT to use" section
above) there's no way to know in advance, per-query, whether reranking will help or hurt without a
ground-truth eval set to check against.


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal BM25 + cross-encoder rerank retrieval + generation, no eval harness."""
from recipes.rerank import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"text": ..., ...}, ...} -- fill in your own chunks
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
```
